In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
from sklearn.decomposition import PCA
import os

warnings.filterwarnings('ignore')

# Проверка GPU (CUDA) или Apple Silicon (MPS), иначе используем CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(f"Используемое устройство: {device}")

In [5]:
# 1. Загрузка данных
print("Загрузка технических фичей...")
df_tech = pd.read_parquet("../data/processed/gbpusd_with_all_features.parquet")
df_tech.index = pd.to_datetime(df_tech.index)

if df_tech.index.tz is not None:
    df_tech.index = df_tech.index.tz_localize(None)
df_tech.sort_index(inplace=True)

print("Загрузка макро-векторов...")
df_macro = pd.read_parquet("../data/processed/sentiment_embeddings.parquet")
df_macro.index = pd.to_datetime(df_macro.index)

if df_macro.index.tz is not None:
    df_macro.index = df_macro.index.tz_localize(None)
df_macro.sort_index(inplace=True)

# 2. Объединение
df_combined = pd.merge_asof(
    df_tech, 
    df_macro, 
    left_index=True, 
    right_index=True, 
    direction='backward'
)

# 3. Разворачиваем 384-мерный вектор в отдельные колонки
print("Разворачивание векторов...")
df_combined['sentiment_vector'] = df_combined['sentiment_vector'].apply(
    lambda x: x if isinstance(x, (np.ndarray, list)) else np.zeros(384)
)

vector_df = pd.DataFrame(df_combined['sentiment_vector'].tolist(), index=df_combined.index)
vector_df.columns = [f'macro_emb_{i}' for i in range(384)]

df_combined = df_combined.drop(columns=['sentiment_vector']).join(vector_df)
df_combined.dropna(inplace=True) 

# ПРОВЕРКА ТЕПЕРЬ ЗДЕСЬ (После разворачивания)
print("Проверка колонок в df_combined:")
print("macro_emb_0 присутствует:", 'macro_emb_0' in df_combined.columns)
print(f"Размерность итогового датасета: {df_combined.shape}")

Проверка колонок в df_combined:
macro_emb_0 присутствует: True
Размерность итогового датасета: (644925, 452)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# 1. Берем полный макро-цикл (с 2000 года)
df_combined = df_combined.loc['2000-01-01':].copy()

SEQ_LEN = 32          
FORWARD_HORIZON = 4   

# 2. Создаем Target
df_combined['future_return'] = np.log(df_combined['close'].shift(-FORWARD_HORIZON) / df_combined['close'])

# ==========================================
# 12 ЭЛИТНЫХ ФИЧЕЙ (После аудита Gemma)
# ==========================================
top_tech_features = [
    "rsi", "premium_discount", "dist_to_liq_high", "dist_to_liq_low", 
    "struct_trend", "Norm_Slope", "dist_to_pdh", "fvg_bear_size", 
    "fvg_bull_size", "dist_to_pdl", "trend_1h", "mtfa_score"
]
macro_cols = [f'macro_emb_{i}' for i in range(384)]

# Очищаем данные от мусора до PCA
feature_cols = top_tech_features + macro_cols
df_clean = df_combined[['future_return'] + feature_cols].copy()
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
df_clean.dropna(inplace=True)

targets = (df_clean['future_return'] > 0).astype(int).values

# 3. ПОИСК СОБЫТИЙ (Event-Based Sampling)
macro_changes = df_clean['macro_emb_0'].diff().abs() > 0
macro_changes.iloc[0] = True 
event_indices = np.where(macro_changes)[0]

# 4. СЖАТИЕ МАКРО-ДАННЫХ (Строго 384 колонки)
print("Сжатие 384 макро-фичей в 8 главных компонент (PCA)...")
pca = PCA(n_components=8)
macro_compressed = pca.fit_transform(df_clean[macro_cols].values) # Изолировали только макро!

# 5. Сборка и Масштабирование (12 тех + 8 макро = 20 колонок)
tech_features_array = df_clean[top_tech_features].values
combined_features = np.hstack([tech_features_array, macro_compressed])

scaler = StandardScaler()
scaled_features = scaler.fit_transform(combined_features)

# 6. Нарезка 3D-окон ТОЛЬКО вокруг новостей
X_events, y_events = [], []
for idx in event_indices:
    if idx >= SEQ_LEN and idx < len(scaled_features) - FORWARD_HORIZON:
        X_events.append(scaled_features[idx - SEQ_LEN : idx])
        y_events.append(targets[idx])

X_events = np.array(X_events)
y_events = np.array(y_events)

split_idx = int(len(X_events) * 0.8)
train_features, train_targets = X_events[:split_idx], y_events[:split_idx]
test_features, test_targets = X_events[split_idx:], y_events[split_idx:]

df_combined.to_parquet("../data/processed/full_merged_dataset.parquet")
print("✅ Полный датасет (с макро-колонками) успешно сохранен в корневую data/processed/!")

print(f"Train: {len(train_features)} событий | Test:  {len(test_features)} событий")
print(f"Размерность: {train_features.shape[2]} фичей (12 тех + 8 макро)")

In [ ]:
class QuantEventDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        # Для BCE Loss нужен формат float32
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Батчи делаем меньше (32), так как данных стало меньше
train_loader = DataLoader(QuantEventDataset(train_features, train_targets), batch_size=32, shuffle=True)
test_loader = DataLoader(QuantEventDataset(test_features, test_targets), batch_size=32, shuffle=False)

class Quantformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2):
        super(Quantformer, self).__init__()
        self.feature_projection = nn.Linear(num_features, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, 
            batch_first=True, dropout=0.2 # Увеличили dropout для защиты от переобучения
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc_out = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1) # Выдает 1 число (Logit)
        )

    def forward(self, x):
        x = self.feature_projection(x)
        trans_out = self.transformer(x)
        last_step_out = trans_out[:, -1, :] 
        return self.fc_out(last_step_out)

model = Quantformer(num_features=train_features.shape[2]).to(device)

# МАГИЯ ЗДЕСЬ: BCEWithLogitsLoss - стандарт для бинарной классификации
criterion = nn.BCEWithLogitsLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005) # Чуть уменьшили learning rate

print(f"Модель Классификации загружена на {device}!")

In [ ]:
EPOCHS = 15
train_losses, test_losses = [], []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Валидация (Тест)
    model.eval()
    running_test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            running_test_loss += loss.item()
            
            # Считаем Hit Rate (Сигмоида переводит Logit в вероятность)
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
            
    avg_test_loss = running_test_loss / len(test_loader)
    test_losses.append(avg_test_loss)
    hit_rate = correct / total
    test_accuracies.append(hit_rate)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Test Hit Rate: {hit_rate:.2%}")

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_title("Функция потерь (BCE)")
ax1.legend()

ax2.plot(test_accuracies, label='Hit Rate (Accuracy)', color='green')
ax2.axhline(0.5, color='red', linestyle='--')
ax2.set_title("Точность предсказания направления (Вверх/Вниз)")
ax2.legend()
plt.show()

In [ ]:
model.eval()
all_probs = []
all_actuals = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        # Для классификации обязательно пропускаем через Сигмоиду
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_actuals.extend(batch_y.cpu().numpy().flatten())

# Превращаем вероятности в классы: >= 50% это 1 (Лонг), иначе 0 (Шорт)
preds_class = (np.array(all_probs) >= 0.5).astype(int)
actuals_class = np.array(all_actuals).astype(int)

hit_rate = np.mean(preds_class == actuals_class)
print(f"РЕАЛЬНАЯ ТОЧНОСТЬ (Hit Rate): {hit_rate:.2%}")

# Дополнительно: посмотрим, насколько модель уверена в себе
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(actuals_class, preds_class)
print(f"Матрица ошибок (Confusion Matrix):\n{cm}")
print("Строки - реальность (0, 1), Колонки - предсказания (0, 1)")

In [ ]:
import torch
import joblib
import os

# Поднимаемся на уровень выше (..) и идем в data/processed
target_dir = "../data/processed"
os.makedirs(target_dir, exist_ok=True)

# Сохраняем по новому пути
torch.save(model.state_dict(), f"{target_dir}/us_quantformer.pth")
joblib.dump(scaler, f"{target_dir}/us_scaler.pkl")
joblib.dump(pca, f"{target_dir}/us_pca.pkl")

print(f"✅ Теперь файлы точно в: {os.path.abspath(target_dir)}")